In [ ]:
from pathlib import Path
import polars as pl
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / "data"
df = pl.read_parquet(data_dir / "interests_clusters/cm0i27jdj0000aqpa73ghpcxf.snappy")
d = show(df.to_pandas())
d.open_browser()

In [ ]:
import numpy as np
i,c = np.unique(df.get_column("cluster_category").fill_null(""), return_counts=True)
for pair in zip(i, c):
    print(pair[1], pair[0])

In [ ]:
import csv
import importlib.util
import json
import subprocess

import jsonpickle
from huggingface_hub import scan_cache_dir


def nvsmi_csv_to_json(csv_string):
    # Split the CSV string into lines
    csv_lines = csv_string.strip().split("\n")
    # Read the CSV data
    reader = csv.DictReader(csv_lines)
    # Convert the CSV data to a list of dictionaries, cleaning keys and values
    data = [
        {k.strip().replace(" ", "_").lower(): v.strip() for k, v in row.items()}
        for row in reader
    ]
    # Convert to JSON
    return data


MIN_CUDA_COMPUTE_CAPABILITY = 7.0


def gpu_info(return_list=False):
    try:
        csv_string = subprocess.run(
            "nvidia-smi --query-gpu=timestamp,name,pci.bus_id,driver_version,pstate,pcie.link.gen.max,pcie.link.gen.current,temperature.gpu,utilization.gpu,utilization.memory,memory.total,memory.free,memory.used --format=csv",
            shell=True,
            text=True,
            capture_output=True,
        ).stdout.strip()
        
        data = nvsmi_csv_to_json(csv_string)

        if return_list:
            return data
        else:
            return json.dumps(data, indent=2)
    except Exception:
        return []


def is_package_installed(package_name):
    spec = importlib.util.find_spec(package_name)
    return spec is not None


def get_cuda_version():
    try:
        cuda_version = subprocess.run(
            "nvcc --version",
            shell=True,
            text=True,
            capture_output=True,
        ).stdout.strip()
        return cuda_version
    except Exception:
        return None


def is_cuda_available():
    try:
        compute_capability = subprocess.run(
            "nvidia-smi --query-gpu=compute_cap --format=csv,noheader|head -n 1",
            shell=True,
            text=True,
            capture_output=True,
        ).stdout.strip()
        return float(compute_capability) >= MIN_CUDA_COMPUTE_CAPABILITY
    except Exception:
        return False


def is_vllm_image():
    return is_package_installed("torch") and is_cuda_available()


def is_rapids_image():
    return is_package_installed("cudf") and is_cuda_available()


def get_hf_cache_info():
    cache_info = scan_cache_dir()

    # Configure jsonpickle to produce human-readable output
    jsonpickle.set_encoder_options("json", indent=2, sort_keys=True)

    # Use jsonpickle to serialize the entire cache_info object
    serialized_cache_info = jsonpickle.encode(cache_info)

    return serialized_cache_info


In [ ]:
subprocess.run(
    "nvidia-smi --query-gpu=timestamp,name,pci.bus_id,driver_version,pstate,pcie.link.gen.max,pcie.link.gen.current,temperature.gpu,utilization.gpu,utilization.memory,memory.total,memory.free,memory.used --format=csv",
    shell=True,
    text=True,
    capture_output=True,
).stdout.strip()

In [ ]:
from textwrap import dedent


interests_clusters = df


def get_categorization_prompt(cluster_interests: str) -> str:
    return dedent(f"""
    Analyze this list and identify up to 5 main categories that best describe the contents of the list.
    For each main category, provide up to 3 sub categories that belong to it also found in the list.                                      
      
    {cluster_interests}
    """).strip()

sampled_df = (
    interests_clusters.filter(pl.col("merged_cluster_label") != -1)
    .group_by("merged_cluster_label")
    .apply(lambda group: group.sample(n=min(len(group), 50)))
)

# Group interests by merged_cluster_label
grouped_interests = (
    sampled_df.group_by("merged_cluster_label")
    .agg(pl.col("interests").alias("cluster_interests"))
    .sort("merged_cluster_label")
)

# Prepare prompts for each merged cluster
prompt_sequences = [
    get_categorization_prompt("\n".join(row["cluster_interests"]))
    for row in grouped_interests.to_dicts()
]

In [ ]:
print(prompt_sequences[8])

In [ ]:
COL = "merged_cluster_label"

df = df.sort("merged_cluster_label", "date", descending=[False, False])

res = df.with_columns(
        pl.concat_str(
            [
                pl.when(pl.col("interests_quirkiness").eq(True))
                .then(pl.lit("NB:"))
                .otherwise(pl.lit("")),
                pl.col("date"),
                pl.lit(":"),
                pl.col("interests"),
            ],
        ).alias("date_interests")
).group_by(
    COL
).agg(
    # Aggregate the interests with their dates into a list of strings
    pl.col("date_interests").str.concat("\n").alias("cluster_items"),
).filter(
    pl.col(COL) == 81
).get_column("cluster_items")[0]

print(res)

In [ ]:
import umap
import plotly.express as px

In [ ]:
df = df.filter(pl.col("fine_centroid").is_not_null()).group_by("fine_dissimilarity_rank").agg(pl.col("fine_centroid").first())

# Reduce dimensionality of fine_centroid embeddings
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(df['fine_centroid'].to_list())

# Create a new dataframe with reduced embeddings and labels
plot_df = pl.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'rank': df['fine_dissimilarity_rank']
})

# Create an interactive scatter plot
fig = px.scatter(
    plot_df.to_pandas(),
    x='x',
    y='y',
    color='rank',
    hover_data=['rank'],
    title='Fine Centroid Embeddings Visualization'
)

# Show the plot
fig.show()

In [ ]:
from sklearn.metrics import pairwise_distances
import numpy as np

# Create some sample data
X = np.array([[-9999, -9999], [1, 1], [-1, -1], [9999, 9999]])

# Calculate pairwise distances using Euclidean distance (default metric)
distances = pairwise_distances(X, metric='cosine')

print("Pairwise distances:")
print(distances)

# Calculate pairwise distances using Manhattan distance
manhattan_distances = pairwise_distances(X, metric='manhattan')

print("\nPairwise Manhattan distances:")
print(manhattan_distances)